# UC-CAT-3 — Schema Evolution: Add a Column

**Persona:** Data Engineer

**Goal:** Safely add a `cloud_cover_pct` column to an existing PostgreSQL-backed collection
using the schema evolution API:
1. Create a test collection with PostgreSQL driver configured
2. Inspect existing backups
3. Dry-run the evolution to inspect the diff without mutating state
4. Apply the evolution
5. Verify the new field appears in the OGC queryables endpoint
6. Cleanup the test collection

**Prerequisites:**
- A catalog already exists (pass `CATALOG_ID` or it defaults to `demo-catalog`)
- Admin JWT in `DYNASTORE_ADMIN_TOKEN`
- `driver:postgresql` installed on the platform

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`, `CATALOG_ID` (optional)

In [ ]:
import os
import json
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN = os.environ.get("DYNASTORE_ADMIN_TOKEN", "") or os.environ.get("DYNASTORE_SYSADMIN_TOKEN", "")
# Use a notebook-unique catalog so the notebook is fully self-contained
CATALOG_ID = f"schema-evo-test-{uuid.uuid4().hex[:8]}"

admin_headers = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=admin_headers, timeout=60.0)
print(f"Connected to {BASE_URL}  catalog={CATALOG_ID}")

# Create the test catalog
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": "Schema Evolution Test Catalog",
    "description": "Ephemeral catalog for schema evolution demo.",
    "stac_version": "1.0.0",
}
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
assert r.status_code == 201, f"Failed to create catalog: {r.status_code}: {r.text}"
print(f"Created catalog: {CATALOG_ID}")

## Setup — Create test collection with PostgreSQL driver

Before schema evolution, we need a collection with PostgreSQL storage configured.
This cell creates it automatically so the notebook is self-contained.

In [ ]:
import uuid

COLLECTION_ID = f"schema-evolution-test-{uuid.uuid4().hex[:8]}"

# Ensure the catalog has a default routing config for collection creation.
catalog_defaults = {
    "configs": {
        "ItemsPostgresqlDriverConfig": {"enabled": True, "collection_type": "VECTOR"},
        "CollectionRoutingConfig": {
            "enabled": True,
            "operations": {
                "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
                "READ":  [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
            },
        },
    }
}
r_bulk = client.put(f"/configs/catalogs/{CATALOG_ID}/bulk", content=json.dumps(catalog_defaults), timeout=60.0)
print(f"Catalog defaults: {r_bulk.status_code}")

# Create the test collection
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": "Schema Evolution Test Collection",
    "description": "Test collection for schema evolution demo.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections", content=json.dumps(collection_payload))
assert r.status_code == 201, f"Failed to create collection: {r.status_code}: {r.text}"
print(f"Created test collection: {COLLECTION_ID}")

## Step 0 — Guard: confirm no reindex is running

Applying a schema evolution while an active reindex is in progress can produce inconsistent
index mappings. Always check before evolving.

In [ ]:
# Query the catalog task log for active REINDEX events
r = client.get(
    f"/logs/catalogs/{CATALOG_ID}/logs",
    params={"event_type": "REINDEX", "level": "INFO", "limit": 10},
)
print(r.status_code, r.text[:500])
# 200 expected; 404 is acceptable if no log backend is configured
assert r.status_code in (200, 404), (
    f"Unexpected status {r.status_code}: {r.text}"
)

if r.status_code == 200:
    log_entries = r.json()
    active_reindex = [
        e for e in (log_entries if isinstance(log_entries, list) else log_entries.get("entries", []))
        if e.get("status") in ("running", "accepted", "RUNNING", "ACCEPTED")
    ]
    assert len(active_reindex) == 0, (
        f"Active reindex detected — do not evolve schema while reindex is running: {active_reindex}"
    )
    print(f"No active reindex found ({len(log_entries if isinstance(log_entries, list) else log_entries.get('entries', []))} recent log entries checked). Safe to proceed.")
else:
    print("Log backend not available (404). Proceeding without reindex guard — verify manually.")

## Step 1 — Check existing backups

The evolution API creates a backup snapshot before applying changes. Listing existing backups
lets you track the history of schema changes and identify a restore point if needed.

In [ ]:
r = client.get(f"/admin/schemas/{CATALOG_ID}/{COLLECTION_ID}/backups")
print(r.status_code, r.text[:600])
# NOTE: /admin/schemas/* endpoints are planned for v0.5.x; 404 is expected until implemented.
if r.status_code == 200:
    backups = r.json()
    backup_list = backups if isinstance(backups, list) else backups.get("backups", [])
    print(f"Existing backups: {len(backup_list)}")
    for b in backup_list:
        print(f"  - {b.get('name', b.get('id', '?'))}  created={b.get('created_at', '?')}")
else:
    print(f"NOTE: Schema evolution API not yet implemented ({r.status_code}). Feature planned for v0.5.x.")

## Step 2 — Dry-run evolution (inspect diff)

Post an `EvolutionRequest` with `apply_safe=false` (or the platform-equivalent dry-run flag).
The server returns the diff of what *would* be applied without mutating the schema.
Always review the diff before applying.

In [ ]:
evolution_request = {
    "fields": [
        {
            "name": "cloud_cover_pct",
            "type": "numeric",
            "nullable": True,
            "description": "Cloud cover percentage [0-100] derived from MODIS QA band.",
        }
    ],
    "apply_safe": False,  # dry-run: compute diff, do not mutate
}

r = client.post(
    f"/admin/schemas/{CATALOG_ID}/{COLLECTION_ID}/evolve",
    content=json.dumps(evolution_request),
)
print(r.status_code, r.text[:800])
# NOTE: /admin/schemas/*/evolve is planned for v0.5.x; 404 is expected until implemented.
if r.status_code == 200:
    dry_run_result = r.json()
    print("\n--- Dry-run diff ---")
    print(json.dumps(dry_run_result, indent=2))
else:
    print(f"NOTE: Schema evolution API not yet implemented ({r.status_code}). Feature planned for v0.5.x.")

## Step 3 — Apply evolution

Re-post with `apply_safe=true`. The platform will:
1. Create a schema backup
2. Execute the `ALTER TABLE ... ADD COLUMN` DDL inside a transaction
3. Return a summary of applied changes

Only backwards-compatible operations (ADD COLUMN, CREATE INDEX) are allowed when `apply_safe=true`.
Destructive operations (DROP COLUMN, ALTER TYPE) are blocked.

In [ ]:
evolution_request_apply = {
    **evolution_request,
    "apply_safe": True,  # apply: mutate schema
}

r = client.post(
    f"/admin/schemas/{CATALOG_ID}/{COLLECTION_ID}/evolve",
    content=json.dumps(evolution_request_apply),
)
print(r.status_code, r.text[:800])
# NOTE: /admin/schemas/*/evolve is planned for v0.5.x; 404 is expected until implemented.
if r.status_code == 200:
    apply_result = r.json()
    print("\n--- Applied changes ---")
    print(json.dumps(apply_result, indent=2))
    result_str = json.dumps(apply_result)
    assert "cloud_cover_pct" in result_str, (
        f"'cloud_cover_pct' not found in apply response: {result_str[:400]}"
    )
    print("'cloud_cover_pct' confirmed in applied changes.")
else:
    print(f"NOTE: Schema evolution API not yet implemented ({r.status_code}). Feature planned for v0.5.x.")

## Step 4 — Verify via queryables

The OGC API Queryables endpoint reflects the current physical schema. After evolution,
`cloud_cover_pct` must appear as a queryable property clients can use in CQL2 filter expressions.

In [ ]:
r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables"
)
print(r.status_code, r.text[:800])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

queryables = r.json()
properties = queryables.get("properties", {})
queryable_names = list(properties.keys())
print(f"Queryable properties ({len(queryable_names)}): {queryable_names}")

# After schema evolution, cloud_cover_pct would appear here.
# Until /admin/schemas/*/evolve is implemented, we just verify the queryables endpoint works.
if "cloud_cover_pct" in queryable_names:
    print("'cloud_cover_pct' is a queryable field — schema evolution confirmed.")
else:
    print("Queryables endpoint works. 'cloud_cover_pct' will appear here after evolution API is implemented.")

## Edge Cases

### Destructive operations are blocked by `apply_safe=true`

The platform rejects DROP COLUMN when `apply_safe=true`. The cell below documents this
constraint without executing a destructive operation — you would need to temporarily
set `apply_safe=false` (which bypasses the guard) to perform irreversible DDL.

In [ ]:
# Document: DROP COLUMN with apply_safe=True must be rejected
drop_request = {
    "fields": [{"name": "cloud_cover_pct", "action": "drop"}],
    "apply_safe": True,
}

r = client.post(
    f"/admin/schemas/{CATALOG_ID}/{COLLECTION_ID}/evolve",
    content=json.dumps(drop_request),
)
print(f"DROP COLUMN with apply_safe=True: status={r.status_code}")
print(r.text[:400])

# NOTE: /admin/schemas/*/evolve is planned for v0.5.x; 404 expected until implemented.
if r.status_code in (400, 409, 422):
    print(f"Confirmed: destructive evolution correctly rejected with {r.status_code}.")
else:
    print(f"NOTE: API not yet implemented ({r.status_code}). When implemented, DROP COLUMN under apply_safe=True must return 4xx.")

## Notes on reindex ordering

- **Always verify** that no reindex is running before applying schema evolution (Step 0 above).
- After evolution, if the collection is indexed in Elasticsearch, the ES mapping must be
  updated separately — schema evolution only alters the PostgreSQL table. Trigger a
  reindex via `POST /search/catalogs/{CATALOG_ID}/reindex` once the column is populated.
- Column additions are `NOT NULL` safe only if a default value is provided or the column
  is nullable. The example above uses `nullable: true` to avoid a full table rewrite.
- There is no teardown cell for this notebook because the column addition is permanent.
  If you need to remove the column, use the admin SQL console with `apply_safe=false`
  and expect a full reindex to follow.

**To clean up the collection entirely**, run the teardown cell in
`catalog/02_create_collection_with_layerconfig.ipynb`.

In [ ]:
print("Session will be closed after teardown.")

## Teardown

Clean up the test collection created in the setup cell above.

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
print(f"Collection delete: {r.status_code}")
# 204 = deleted; 404 = already gone (notebook re-run); both are acceptable
assert r.status_code in (204, 404), f"Unexpected status: {r.status_code}: {r.text}"
print(f"Test collection {COLLECTION_ID} cleaned up.")

r2 = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"})
assert r2.status_code == 204, f"Failed to delete catalog: {r2.status_code}: {r2.text}"
print(f"Test catalog {CATALOG_ID} deleted.")
client.close()
print("Session closed.")